In [5]:
# embed_and_save_csv_from_rar.py
import os
import pickle
import numpy as np
import pandas as pd
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
import rarfile

# For Google Colab file upload and download
try:
    from google.colab import files
except ImportError:
    files = None

# ----------------- CSV Loader (with column selection & row sampling) -----------------
def load_csv_documents(file_path, columns_to_use=None, sample_fraction=1.0):
    """
    Load CSV and convert each row into a single text document for RAG.
    Ignores 'product_price' column and optionally selects only a subset of columns.
    Can also sample only a fraction of rows.

    Args:
        file_path (str): Path to CSV
        columns_to_use (list[str]): List of column names to include in the document
        sample_fraction (float): Fraction of rows to use (0 < sample_fraction <= 1)
    """
    df = pd.read_csv(file_path)

    # Sample rows if requested
    if sample_fraction < 1.0:
        df = df.sample(frac=sample_fraction, random_state=42).reset_index(drop=True)
        print(f"[INFO] Sampled {len(df)} rows ({sample_fraction*100:.0f}% of total)")

    # Default columns if none provided
    if columns_to_use is None:
        columns_to_use = [
            'product_name', 'sub_category', 'salt_composition',
            'product_manufactured', 'medicine_desc',
            'side_effects', 'drug_interactions'
        ]

    # Ensure columns exist in the CSV
    columns_to_use = [col for col in columns_to_use if col in df.columns]

    documents = []
    for _, row in df.iterrows():
        desc_parts = []
        for col in columns_to_use:
            desc_parts.append(f"{col.replace('_', ' ').title()}: {row[col]}")
        documents.append("\n".join(desc_parts))

    return documents

# ----------------- Embedding & Caching -----------------
EMBEDDINGS_FILE = "doc_embeddings.npy"
DOCS_FILE = "documents.pkl"

print("[INFO] Loading embedding model...")
model = SentenceTransformer('all-MiniLM-L6-v2')
print("[INFO] Embedding model loaded.")

def embed_text(text):
    """Return embedding vector for a single text document"""
    return model.encode(text)

def save_documents_and_embeddings(documents, embeddings):
    """Save documents and embeddings locally"""
    np.save(EMBEDDINGS_FILE, embeddings)
    with open(DOCS_FILE, "wb") as f:
        pickle.dump(documents, f)
    print(f"[INFO] Saved {len(documents)} documents and embeddings locally.")

    # If in Colab, offer download
    if files:
        files.download(EMBEDDINGS_FILE)
        files.download(DOCS_FILE)
        print("[INFO] Downloaded embeddings and documents to local machine.")

def load_documents_and_embeddings():
    """Load documents and embeddings if they exist"""
    if os.path.exists(EMBEDDINGS_FILE) and os.path.exists(DOCS_FILE):
        print("[INFO] Loading cached documents and embeddings...")
        embeddings = np.load(EMBEDDINGS_FILE)
        with open(DOCS_FILE, "rb") as f:
            documents = pickle.load(f)
        print(f"[INFO] Loaded {len(documents)} documents from cache.")
        return documents, embeddings
    return None, None

def extract_csv_from_rar(rar_path):
    """Extract CSV from RAR file and return CSV path"""
    rf = rarfile.RarFile(rar_path)
    csv_files = [f for f in rf.namelist() if f.lower().endswith(".csv")]
    if not csv_files:
        raise FileNotFoundError("No CSV file found in the RAR archive.")
    csv_path = csv_files[0]
    rf.extract(csv_path)
    print(f"[INFO] Extracted CSV: {csv_path}")
    return csv_path

def embed_csv_from_rar(rar_path=None, sample_fraction=1.0, columns_to_use=None):
    """Load CSV from RAR, embed all documents (or sampled fraction), caching results"""
    # If no RAR path provided and in Colab, upload RAR
    if rar_path is None and files:
        uploaded = files.upload()
        rar_path = list(uploaded.keys())[0]
        print(f"[INFO] Uploaded RAR: {rar_path}")

    # Check cache first
    cached_docs, cached_embs = load_documents_and_embeddings()
    if cached_docs is not None and cached_embs is not None:
        return cached_docs, cached_embs

    # Extract CSV from RAR
    print("[INFO] Extracting CSV from RAR...")
    csv_path = extract_csv_from_rar(rar_path)

    # Load documents with optional row sampling
    print("[INFO] Loading CSV documents...")
    documents = load_csv_documents(csv_path, columns_to_use=columns_to_use, sample_fraction=sample_fraction)
    print(f"[INFO] Loaded {len(documents)} documents.")

    # Embed with progress bar
    print("[INFO] Computing embeddings for all documents...")
    embeddings = []
    for doc in tqdm(documents, desc="Embedding documents"):
        embeddings.append(embed_text(doc))
    embeddings = np.array(embeddings)
    print("[INFO] All documents embedded.")

    # Save locally and optionally download
    save_documents_and_embeddings(documents, embeddings)

    return documents, embeddings

# ----------------- Example usage -----------------
if __name__ == "__main__":
    # Only use 30% of the CSV rows
    docs, doc_embeddings = embed_csv_from_rar(sample_fraction=0.3)
    print("[INFO] Embedding pipeline completed successfully.")


[INFO] Loading embedding model...
[INFO] Embedding model loaded.


Saving medicine_data.rar to medicine_data.rar
[INFO] Uploaded RAR: medicine_data.rar
[INFO] Extracting CSV from RAR...
[INFO] Extracted CSV: medicine_data.csv
[INFO] Loading CSV documents...
[INFO] Sampled 58682 rows (30% of total)
[INFO] Loaded 58682 documents.
[INFO] Computing embeddings for all documents...


Embedding documents: 100%|██████████| 58682/58682 [08:03<00:00, 121.47it/s]


[INFO] All documents embedded.
[INFO] Saved 58682 documents and embeddings locally.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

[INFO] Downloaded embeddings and documents to local machine.
[INFO] Embedding pipeline completed successfully.
